In [1]:
## Load environment variables

from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

print("Groq key loaded:", bool(api_key))

Groq key loaded: True


In [2]:
## Initialize ChatGroq

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_retries=2,
    reasoning_format="hidden"
)

In [3]:
## First basic LLM test

response = llm.invoke(
    "Reply with exactly: Groq connection successful."
)

print(response.content)

Groq connection successful


In [4]:
print(type(response))

<class 'langchain_core.messages.ai.AIMessage'>


In [5]:
## Create the RAG prompt

from langchain_core.prompts import ChatPromptTemplate

In [6]:
rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a Credit Risk Knowledge Assistant.

Your job is to answer questions using ONLY the supplied context.

Rules:

1. Use only information present in the context.
2. Do not use outside knowledge to answer the question.
3. If the context does not contain enough information, say:
   "I could not find enough information in the provided documents."
4. Do not invent regulatory requirements, numbers, dates, thresholds,
   classifications, or definitions.
5. Give a clear and concise answer.
6. When possible, mention the supporting source and page supplied
   in the context.
7. If multiple sources provide relevant information, distinguish them clearly.
"""
        ),
        (
            "human",
            """
Context:
{context}

Question:
{question}

Answer:
"""
        ),
    ]
)

In [8]:
## Test the prompt manually first

test_context = """
Source: INDAS109.pdf
Page: 1

The objective of this Standard is to establish principles for the
financial reporting of financial assets and financial liabilities that
will present relevant and useful information to users of financial
statements for their assessment of the amounts, timing and uncertainty
of an entity's future cash flows.
"""

question = "What is the objective of Ind AS 109?"

In [9]:
messages = rag_prompt.invoke(
    {
        "context": test_context,
        "question": question
    }
)

response = llm.invoke(messages)

print(response.content)

The objective of Ind AS 109 is to establish principles for the financial reporting of financial assets and financial liabilities that will present relevant and useful information to users of financial statements for their assessment of the amounts, timing and uncertainty of an entity’s future cash flows【INDAS109.pdf, p. 1】.


In [10]:
## Test the hallucination guard

question = "Who is the CEO of Microsoft?"

messages = rag_prompt.invoke(
    {
        "context": test_context,
        "question": question
    }
)

response = llm.invoke(messages)

print(response.content)

I could not find enough information in the provided documents.


In [13]:
import sys
from pathlib import Path

# Main Project_1 folder
PROJECT_ROOT = Path(
    r"D:\AI Projects\Credit-Risk-Knowledge-Assistant"
)

# Add project root to Python's import search path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root added:")
print(PROJECT_ROOT)

Project root added:
D:\AI Projects\Credit-Risk-Knowledge-Assistant


In [14]:
## Connect the real retriever

from app.services.retriever_service import (
    retrieve_documents
)

In [15]:
question = (
    "What objective is stated in Chapter 1 "
    "of Ind AS 109 Financial Instruments?"
)

retrieved_documents = retrieve_documents(
    question=question,
    search_type="similarity",
    top_k=5
)

print(
    "Retrieved documents:",
    len(retrieved_documents)
)

d:\AI Projects\Credit-Risk-Knowledge-Assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15609.67it/s]


Retrieved documents: 5


In [16]:
## Convert Documents into prompt context

def format_context(documents):

    formatted_chunks = []

    for document in documents:

        source = document.metadata.get(
            "source_file",
            "Unknown"
        )

        page = document.metadata.get(
            "page_number",
            "Unknown"
        )

        text = document.page_content

        formatted_chunk = (
            f"Source: {source}\n"
            f"Page: {page}\n"
            f"Content:\n{text}"
        )

        formatted_chunks.append(
            formatted_chunk
        )

    return "\n\n---\n\n".join(
        formatted_chunks
    )


In [17]:
context = format_context(
    retrieved_documents
)

print(context[:2000])

Source: INDAS109.pdf
Page: 1
Content:
246 
 
 
Indian Accounting Standard (Ind AS) 109 
Financial Instruments 
 
(The Indian Accounting Standard includes paragraphs set in bold type and plain 
type, which have equal authority. Paragraphs in bold type indicate the main 
principles.) 
 
 
Chapter 1 Objective 
 
1.1 The objective of this Standard is to establish principles for the 
financial reporting of financial assets and financial liabilities that will 
present relevant and useful information to users of financial 
statements for their assessment of the amounts, timi ng and uncertainty 
of an entity’s future cash flows.  
 
Chapter 2 Scope 
 
2.1 This Standard shall be applied by all entities to all types of 
financial instruments except:  
 
 
(a) those interests in subsidiaries, associates and joint ventures 
that are accounted for in accordance with Ind AS1 10 
ConsolidatedFinancial Statements , I nd AS 27  Separate 
Financial Statements orInd AS 28 Investments in Associates

---



In [18]:
## Generate answer using real retrieval

messages = rag_prompt.invoke(
    {
        "context": context,
        "question": question
    }
)

response = llm.invoke(messages)

print(response.content)

The objective stated in Chapter 1 of Ind AS 109 is:

> “The objective of this Standard is to establish principles for the financial reporting of financial assets and financial liabilities that will present relevant and useful information to users of financial statements for their assessment of the amounts, timing and uncertainty of an entity’s future cash flows.”【Source: INDAS109.pdf, Page 1】


In [19]:
## Inspect what the LLM actually received

print(
    messages.to_string()
)

System: 
You are a Credit Risk Knowledge Assistant.

Your job is to answer questions using ONLY the supplied context.

Rules:

1. Use only information present in the context.
2. Do not use outside knowledge to answer the question.
3. If the context does not contain enough information, say:
   "I could not find enough information in the provided documents."
4. Do not invent regulatory requirements, numbers, dates, thresholds,
   classifications, or definitions.
5. Give a clear and concise answer.
6. When possible, mention the supporting source and page supplied
   in the context.
7. If multiple sources provide relevant information, distinguish them clearly.

Human: 
Context:
Source: INDAS109.pdf
Page: 1
Content:
246 
 
 
Indian Accounting Standard (Ind AS) 109 
Financial Instruments 
 
(The Indian Accounting Standard includes paragraphs set in bold type and plain 
type, which have equal authority. Paragraphs in bold type indicate the main 
principles.) 
 
 
Chapter 1 Objective 
 
1.1 Th

In [20]:
## Test several real questions

test_questions = [
    "What objective is stated in Chapter 1 of Ind AS 109 Financial Instruments?",
    "What is expected credit loss?",
    "What is significant increase in credit risk?",
    "What are the rules regarding income recognition for NPAs?",
    "What guidelines apply to bank finance to NBFCs?",
    "What is the capital of France?"
]

In [21]:
for question in test_questions:

    print("\n" + "=" * 100)
    print("QUESTION:")
    print(question)

    retrieved_documents = retrieve_documents(
        question=question,
        top_k=5
    )

    context = format_context(
        retrieved_documents
    )

    messages = rag_prompt.invoke(
        {
            "context": context,
            "question": question
        }
    )

    response = llm.invoke(messages)

    print("\nANSWER:")
    print(response.content)


QUESTION:
What objective is stated in Chapter 1 of Ind AS 109 Financial Instruments?


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10476.35it/s]



ANSWER:
The objective stated in Chapter 1 of Ind AS 109 is:

> “The objective of this Standard is to establish principles for the financial reporting of financial assets and financial liabilities that will present relevant and useful information to users of financial statements for their assessment of the amounts, timing and uncertainty of an entity’s future cash flows.”【Source: INDAS109.pdf, Page 1】

QUESTION:
What is expected credit loss?


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2575.02it/s]



ANSWER:
**Expected credit loss**  
A probability‑weighted estimate of the credit losses that are expected to occur over the expected life of a financial instrument. It is the present value of all cash shortfalls – the difference between the cash flows that are due under the contract and the cash flows that the entity expects to receive. (Source: INDAS109, page 120)  

**12‑month expected credit loss**  
The portion of lifetime expected credit losses that represents the expected losses that would result if a default occurs within 12 months after the reporting date. (Source: INDAS109, page 50)

QUESTION:
What is significant increase in credit risk?


...


ANSWER:
**Rules on income recognition for NPAs**

| Rule | What it says | Source |
|------|--------------|--------|
| **No interest income on NPAs** | Banks must not charge or record interest on any non‑performing asset (NPA), including those that are government‑guaranteed. | Master Circular, §3.1.1 (page 7) |
| **Reversal of previously recognised interest** | If an advance (or any other facility) becomes an NPA, all interest that had been accrued and credited to the income account in earlier periods must be reversed. | Master Circular, §3.2.1 (page 7) |

These provisions ensure that income recognition for NPAs is strictly objective and based on actual recovery records.

QUESTION:
What guidelines apply to bank finance to NBFCs?


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11369.97it/s]



ANSWER:
**Guidelines that govern bank finance to NBFCs**

| Section of the Master Circular (2025) | Key points that apply to bank finance |
|--------------------------------------|---------------------------------------|
| **1. Introduction – Terminology & Background** | Sets the definitions and scope of the circular. |
| **2. Bank Finance to NBFCs registered with RBI** | Banks may extend need‑based working‑capital and term loans to all RBI‑registered NBFCs engaged in infrastructure financing, equipment leasing, hire‑purchase, loan, factoring and investment activities, subject to paragraph 8 of the guidelines. |
| **3. Bank Finance to NBFCs not requiring registration** | Similar credit can be extended to NBFCs that do not need RBI registration, provided they meet the same activity criteria. |
| **4. Activities not eligible for bank credit** | Banks must not finance activities listed in paragraph 4 (e.g., certain types of loans, inter‑corporate deposits, etc.). |
| **5. Bank Finance to

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2261.33it/s]



ANSWER:
I could not find enough information in the provided documents.
